# Phase 1 - SQL Analytics Layer

            **Project:** Hospital Operations & Revenue Risk Intelligence Platform

            **Business goal:** Create a reliable, queryable hospital data layer that
            leadership can trust for operational and financial decision-making.

            This notebook uses the Phase 1 SQLite database created by
            `scripts/build_phase1_database.py`. It verifies the schema, joins,
            indexes, reusable KPI views, business queries, and data-quality checks.

In [ ]:
from pathlib import Path
            import sqlite3
            import pandas as pd

            PROJECT_ROOT = Path.cwd()
            if PROJECT_ROOT.name == "notebooks":
                PROJECT_ROOT = PROJECT_ROOT.parent

            DB_PATH = PROJECT_ROOT / "database" / "hospital_operations.db"
            SQL_QUERY_PATH = PROJECT_ROOT / "sql" / "phase1_analysis_queries.sql"
            DB_PATH

In [ ]:
conn = sqlite3.connect(DB_PATH)
            conn.execute("PRAGMA foreign_keys = ON")

            def sql(query: str) -> pd.DataFrame:
                return pd.read_sql_query(query, conn)

## 1. Database Load Audit

            The raw source files are loaded into staging tables first, then inserted
            into typed relational tables with primary keys, foreign keys, constraints,
            and indexes.

In [ ]:
sql("SELECT * FROM load_audit ORDER BY table_name")

In [ ]:
sql("""
            SELECT 'patients' AS table_name, COUNT(*) AS row_count FROM patients
            UNION ALL
            SELECT 'visits', COUNT(*) FROM visits
            UNION ALL
            SELECT 'billing', COUNT(*) FROM billing
            """)

## 2. Relational Integrity

            These checks verify that visits link to valid patients and billing rows
            link to valid visits.

In [ ]:
sql("PRAGMA foreign_key_check")

In [ ]:
sql("""
            SELECT
                'visits_without_patient' AS check_name,
                COUNT(*) AS issue_count
            FROM visits v
            LEFT JOIN patients p ON p.patient_id = v.patient_id
            WHERE p.patient_id IS NULL
            UNION ALL
            SELECT
                'billing_without_visit',
                COUNT(*)
            FROM billing b
            LEFT JOIN visits v ON v.visit_id = b.visit_id
            WHERE v.visit_id IS NULL
            UNION ALL
            SELECT
                'visits_without_billing',
                COUNT(*)
            FROM visits v
            LEFT JOIN billing b ON b.visit_id = v.visit_id
            WHERE b.visit_id IS NULL
            """)

## 3. Operational Analysis

            These queries support patient flow and departmental efficiency decisions.

In [ ]:
sql("""
            SELECT * FROM v_department_kpis
            ORDER BY total_visits DESC
            """)

In [ ]:
sql("""
            SELECT
                doctor_id,
                COUNT(*) AS high_risk_visits
            FROM visits
            WHERE risk_score = 'High'
            GROUP BY doctor_id
            ORDER BY high_risk_visits DESC
            LIMIT 10
            """)

In [ ]:
sql("""
            SELECT * FROM v_city_patient_flow_kpis
            ORDER BY avg_visits_per_patient DESC
            """)

## 4. Financial Analysis

            These queries support revenue leakage monitoring, insurer behavior
            analysis, payment delay tracking, and realization-ratio reporting.

In [ ]:
sql("""
            SELECT * FROM v_insurance_kpis
            ORDER BY total_billed_amount DESC
            """)

In [ ]:
sql("""
            SELECT
                department,
                total_billed_amount,
                total_approved_amount,
                revenue_realization_ratio
            FROM v_department_kpis
            ORDER BY revenue_realization_ratio DESC
            """)

In [ ]:
sql("""
            WITH ranked_claims AS (
                SELECT
                    b.bill_id,
                    b.visit_id,
                    v.department,
                    p.insurance_provider,
                    b.billed_amount,
                    b.approved_amount,
                    b.claim_status,
                    NTILE(20) OVER (ORDER BY b.billed_amount) AS billed_amount_twentieth
                FROM billing b
                JOIN visits v ON v.visit_id = b.visit_id
                JOIN patients p ON p.patient_id = v.patient_id
            )
            SELECT *
            FROM ranked_claims
            WHERE billed_amount_twentieth = 20
              AND (approved_amount IS NULL OR approved_amount = 0)
            ORDER BY billed_amount DESC
            LIMIT 20
            """)

## 5. Data Quality and Integrity Checks

            These checks are intentionally preserved as reusable SQL views so the same
            logic can be reused in Phase 2 EDA and Phase 6 monitoring.

In [ ]:
sql("SELECT * FROM v_quality_summary")

In [ ]:
sql("""
            SELECT *
            FROM v_quality_temporal_anomalies
            LIMIT 20
            """)

## 6. Index Strategy

            The main indexes support:

            - Department volume, average LOS, and high-risk rate:
              `idx_visits_department`, `idx_visits_department_risk`
            - High-risk doctor workload:
              `idx_visits_doctor_risk`
            - Patient flow by city and provider:
              `idx_patients_city`, `idx_patients_insurance_provider`,
              `idx_visits_patient_id`
            - Claim status, payment delay, and billing amount queries:
              `idx_billing_claim_status`, `idx_billing_payment_days`,
              `idx_billing_billed_amount`, `idx_billing_claim_payment`
            - Time-based downstream modeling and audits:
              `idx_visits_visit_date`, `idx_billing_billing_date`,
              `idx_patients_registration_date`

In [ ]:
sql("""
            SELECT
                name AS index_name,
                tbl_name AS table_name,
                sql AS index_sql
            FROM sqlite_master
            WHERE type = 'index'
              AND sql IS NOT NULL
            ORDER BY table_name, index_name
            """)